In [3]:
import pandas as pd
import numpy as np
import openpyxl
import re

# ── 1. LOAD ALL DATASETS ───────────────────────────────────────────

hh    = pd.read_csv("household_vista_2024_2025.csv")
pers  = pd.read_csv("person_vista_2024_2025.csv")
trips = pd.read_csv("trips_vista_2024_2025.csv")
budget = pd.read_csv("state-budget-2023-24-budget-website-map-data.csv")

# SEIFA: skip first 6 rows of ABS metadata, use row 6 as header
wb = openpyxl.load_workbook("Local Government Area, Indexes, SEIFA 2021.xlsx", read_only=True)
ws = wb["Table 1"]
rows = []
for i, row in enumerate(ws.iter_rows(values_only=True)):
    if i < 6:
        continue
    if row[0] is None:
        break
    rows.append(row[:11])

seifa = pd.DataFrame(rows, columns=[
    "LGA_Code", "LGA_Name",
    "IRSD_Score", "IRSD_Decile",
    "IRSAD_Score", "IRSAD_Decile",
    "IER_Score", "IER_Decile",
    "IEO_Score", "IEO_Decile",
    "Population"
])
seifa["LGA_Code"] = seifa["LGA_Code"].astype(str)

datasets = {
    "Households": hh,
    "Persons":    pers,
    "Trips":      trips,
    "Budget":     budget,
    "SEIFA":      seifa
}

# ── 2. SHAPE AND DTYPES ────────────────────────────────────────────

print("=" * 60)
print("SECTION 1: SHAPE AND COLUMN TYPES")
print("=" * 60)
for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(df.dtypes.to_string())

# ── 3. MISSING VALUES ──────────────────────────────────────────────

print("\n" + "=" * 60)
print("SECTION 2: MISSING VALUES")
print("=" * 60)
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    pct = (missing / len(df) * 100).round(2)
    print(f"\n--- {name} ---")
    if missing.empty:
        print("No missing values found.")
    else:
        result = pd.DataFrame({"Missing Count": missing, "Missing %": pct})
        print(result.to_string())

# ── 4. DUPLICATE ROWS ─────────────────────────────────────────────

print("\n" + "=" * 60)
print("SECTION 3: DUPLICATE ROWS")
print("=" * 60)
for name, df in datasets.items():
    dupes = df.duplicated().sum()
    print(f"{name}: {dupes} duplicate rows")

# ── 5. KEY ID FIELD UNIQUENESS ────────────────────────────────────

print("\n" + "=" * 60)
print("SECTION 4: KEY ID FIELD UNIQUENESS")
print("=" * 60)

print(f"Households - unique hhid: {hh['hhid'].nunique()} / {len(hh)} rows")
print(f"Persons    - unique persid: {pers['persid'].nunique()} / {len(pers)} rows")
print(f"Trips      - unique tripid: {trips['tripid'].nunique()} / {len(trips)} rows")
print(f"SEIFA      - unique LGA_Code: {seifa['LGA_Code'].nunique()} / {len(seifa)} rows")

# ── 6. RELATIONAL JOIN CHECK (VISTA) ──────────────────────────────

print("\n" + "=" * 60)
print("SECTION 5: RELATIONAL JOIN CHECK (VISTA)")
print("=" * 60)

# persons -> households
pers_hh_match = pers['hhid'].isin(hh['hhid']).sum()
print(f"Persons with matching hhid in Households: {pers_hh_match} / {len(pers)}")
print(f"Persons with NO matching hhid: {len(pers) - pers_hh_match}")

# trips -> persons
trips_pers_match = trips['persid'].isin(pers['persid']).sum()
print(f"Trips with matching persid in Persons: {trips_pers_match} / {len(trips)}")
print(f"Trips with NO matching persid: {len(trips) - trips_pers_match}")

# trips -> households
trips_hh_match = trips['hhid'].isin(hh['hhid']).sum()
print(f"Trips with matching hhid in Households: {trips_hh_match} / {len(trips)}")
print(f"Trips with NO matching hhid: {len(trips) - trips_hh_match}")

# ── 7. LGA NAME FORMATS ───────────────────────────────────────────

print("\n" + "=" * 60)
print("SECTION 6: LGA NAME FORMAT COMPARISON")
print("=" * 60)

vista_lgas = sorted(hh['homelga'].dropna().unique())
budget_lgas = sorted(budget['lga'].dropna().unique())
vic_seifa = seifa[seifa['LGA_Code'].str.startswith('2')]
seifa_lgas = sorted(vic_seifa['LGA_Name'].dropna().unique())

print(f"\nVISTA LGA format (sample 5): {vista_lgas[:5]}")
print(f"Budget LGA format (sample 5): {budget_lgas[:5]}")
print(f"SEIFA LGA format (sample 5):  {seifa_lgas[:5]}")

# Check overlap after basic cleaning
def clean_lga(name):
    return re.sub(r'\s*\(.\)', '', str(name)).strip().lower()

vista_clean  = set(clean_lga(l) for l in vista_lgas)
budget_clean = set(clean_lga(l) for l in budget_lgas)
seifa_clean  = set(clean_lga(l) for l in seifa_lgas)

print(f"\nAfter basic cleaning (strip suffixes like (C), (S)):")
print(f"VISTA vs Budget overlap: {len(vista_clean & budget_clean)} LGAs")
print(f"VISTA vs SEIFA overlap:  {len(vista_clean & seifa_clean)} LGAs")

print(f"\nVISTA LGAs NOT in Budget: {sorted(vista_clean - budget_clean)}")
print(f"VISTA LGAs NOT in SEIFA:  {sorted(vista_clean - seifa_clean)}")

# ── 8. NUMERIC RANGE CHECKS ───────────────────────────────────────

print("\n" + "=" * 60)
print("SECTION 7: NUMERIC RANGE CHECKS")
print("=" * 60)

print("\n-- Trips: starthour and arrhour --")
print(f"starthour range: {trips['starthour'].min()} to {trips['starthour'].max()}")
print(f"arrhour range:   {trips['arrhour'].min()} to {trips['arrhour'].max()}")
print(f"starthour values outside 0-27: {trips[~trips['starthour'].between(0,27)]['starthour'].value_counts().to_dict()}")

print("\n-- Trips: triptime and travtime --")
print(f"triptime range: {trips['triptime'].min()} to {trips['triptime'].max()} mins")
print(f"travtime range: {trips['travtime'].min()} to {trips['travtime'].max()} mins")
print(f"Zero or negative triptime: {(trips['triptime'] <= 0).sum()}")

print("\n-- Households: totalvehs --")
print(trips['starthour'].value_counts().sort_index().to_string())

print("\n-- Budget: lat/long --")
print(f"Lat range:  {budget['lat'].min()} to {budget['lat'].max()}")
print(f"Long range: {budget['long'].min()} to {budget['long'].max()}")
print(f"Missing lat: {budget['lat'].isna().sum()}, Missing long: {budget['long'].isna().sum()}")

print("\n-- SEIFA: Score ranges --")
print(f"IRSD_Score range: {seifa['IRSD_Score'].min()} to {seifa['IRSD_Score'].max()}")
print(f"IRSD_Decile range: {seifa['IRSD_Decile'].min()} to {seifa['IRSD_Decile'].max()}")

print("\n" + "=" * 60)
print("STEP 1 COMPLETE - Share this full output for verification")
print("=" * 60)

SECTION 1: SHAPE AND COLUMN TYPES

--- Households ---
Shape: 2486 rows x 28 columns
hhid                              str
surveyperiod                      str
hhsize                          int64
dwelltype                         str
owndwell                          str
totalbikes                        str
totalvehs                       int64
travmonth                         str
travdow                           str
homelga                           str
youngestgroup_5                   str
aveagegroup_5                     str
oldestgroup_5                     str
hhinc_group                       str
hhpoststratweight             float64
hhpoststratweight_GROUP_1     float64
hhpoststratweight_GROUP_2     float64
hhpoststratweight_GROUP_3     float64
hhpoststratweight_GROUP_4     float64
hhpoststratweight_GROUP_5     float64
hhpoststratweight_GROUP_6     float64
hhpoststratweight_GROUP_7     float64
hhpoststratweight_GROUP_8     float64
hhpoststratweight_GROUP_9     float64
hhpo

In [4]:
import pandas as pd
import numpy as np
import openpyxl
import re

# ── RELOAD DATASETS ───────────────────────────────────────────────

hh    = pd.read_csv("household_vista_2024_2025.csv")
pers  = pd.read_csv("person_vista_2024_2025.csv")
trips = pd.read_csv("trips_vista_2024_2025.csv")
budget = pd.read_csv("state-budget-2023-24-budget-website-map-data.csv")

wb = openpyxl.load_workbook("Local Government Area, Indexes, SEIFA 2021.xlsx", read_only=True)
ws = wb["Table 1"]
rows = []
for i, row in enumerate(ws.iter_rows(values_only=True)):
    if i < 6:
        continue
    if row[0] is None:
        break
    rows.append(row[:11])
seifa = pd.DataFrame(rows, columns=[
    "LGA_Code", "LGA_Name",
    "IRSD_Score", "IRSD_Decile",
    "IRSAD_Score", "IRSAD_Decile",
    "IER_Score", "IER_Decile",
    "IEO_Score", "IEO_Decile",
    "Population"
])
seifa["LGA_Code"] = seifa["LGA_Code"].astype(str)
vic_seifa = seifa[seifa["LGA_Code"].str.startswith("2")]

# ── ISSUE 1: hhinc_group missing values ───────────────────────────

print("=" * 60)
print("ISSUE 1: hhinc_group missing values")
print("=" * 60)
missing_inc = hh[hh['hhinc_group'].isna()]
print(f"Rows with missing hhinc_group: {len(missing_inc)}")
print(f"\nAre these missing in the raw file or coded differently?")
print("Unique values in hhinc_group including nulls:")
print(hh['hhinc_group'].value_counts(dropna=False).tail(10))
print(f"\nHousehold sizes of records with missing income:")
print(missing_inc['hhsize'].value_counts().sort_index())
print(f"\nHome regions of records with missing income:")
print(missing_inc['homeregion_ASGS'].value_counts())

# ── ISSUE 2: Missing person weights ──────────────────────────────

print("\n" + "=" * 60)
print("ISSUE 2: 60 persons with missing survey weights")
print("=" * 60)
missing_wt = pers[pers['perspoststratweight'].isna()]
print(f"Count: {len(missing_wt)}")
print(f"\nAge groups of persons with missing weights:")
print(missing_wt['agegroup'].value_counts())
print(f"\nMain activity of persons with missing weights:")
print(missing_wt['mainact'].value_counts())
print(f"\nDo these persons have trips recorded?")
missing_persids = missing_wt['persid'].tolist()
trips_for_missing = trips[trips['persid'].isin(missing_persids)]
print(f"Trips recorded for these 60 persons: {len(trips_for_missing)}")

# ── ISSUE 3: Budget missing lat/long/lga ─────────────────────────

print("\n" + "=" * 60)
print("ISSUE 3: 26 Budget rows missing lga/lat/long")
print("=" * 60)
missing_budget = budget[budget['lat'].isna()]
print(f"Count: {len(missing_budget)}")
print(f"\nTheme breakdown of missing rows:")
print(missing_budget['theme'].value_counts())
print(f"\nFull list of missing rows:")
print(missing_budget[['name','theme','type','investment ','lga']].to_string())

# ── ISSUE 4: String columns that should be numeric ───────────────

print("\n" + "=" * 60)
print("ISSUE 4: String columns that should be numeric")
print("=" * 60)

# totalbikes
print("\n-- Households: totalbikes --")
print(hh['totalbikes'].value_counts(dropna=False).head(15))

# numstops
print("\n-- Persons: numstops --")
print(pers['numstops'].value_counts(dropna=False).head(15))

# cumdist (trips)
print("\n-- Trips: cumdist (sample of unique values) --")
cumdist_vals = trips['cumdist'].dropna().unique()
print(sorted(cumdist_vals)[:20])

# dist1 (trips) - representative of dist1-dist9
print("\n-- Trips: dist1 (sample of unique values) --")
dist1_vals = trips['dist1'].dropna().unique()
print(sorted(dist1_vals)[:20])

# time2 (trips) - representative of time2-time9
print("\n-- Trips: time2 (sample of unique values) --")
time2_vals = trips['time2'].dropna().unique()
print(sorted(time2_vals)[:20])

# investment (budget)
print("\n-- Budget: investment column --")
print(budget['investment '].value_counts(dropna=False).head(20))

# ── ISSUE 5: LGA name mismatches ─────────────────────────────────

print("\n" + "=" * 60)
print("ISSUE 5: LGA name mismatches")
print("=" * 60)

def clean_lga(name):
    return re.sub(r'\s*\(.\w*\)', '', str(name)).strip().lower()

vista_lgas = sorted(hh['homelga'].dropna().unique())
vista_clean = {clean_lga(l): l for l in vista_lgas}

# SEIFA: check full names for the 3 mismatches
print("\nSearching SEIFA for Bayside, Kingston, Merri-bek variants:")
for keyword in ['bayside', 'kingston', 'merri', 'moreland']:
    matches = vic_seifa[vic_seifa['LGA_Name'].str.lower().str.contains(keyword)]
    print(f"  '{keyword}': {matches['LGA_Name'].tolist()}")

# Budget: check for Manningham
print("\nSearching Budget for Manningham:")
manningham = budget[budget['lga'].str.lower().str.contains('manningham', na=False)]
print(f"  Rows found: {len(manningham)}")

# Full LGA mismatch table
budget_clean = set(clean_lga(l) for l in budget['lga'].dropna().unique())
seifa_clean  = set(clean_lga(l) for l in vic_seifa['LGA_Name'].dropna().unique())
vista_set    = set(vista_clean.keys())

print("\nVISTA LGAs and their match status:")
print(f"{'VISTA LGA':<30} {'In Budget?':<15} {'In SEIFA?':<15}")
print("-" * 60)
for lga_clean, lga_orig in sorted(vista_clean.items()):
    in_budget = "YES" if lga_clean in budget_clean else "NO"
    in_seifa  = "YES" if lga_clean in seifa_clean  else "NO"
    print(f"{lga_orig:<30} {in_budget:<15} {in_seifa:<15}")

# ── ISSUE 6: triptime outliers ────────────────────────────────────

print("\n" + "=" * 60)
print("ISSUE 6: Trip time outliers")
print("=" * 60)
print(f"\ntriptime distribution:")
print(trips['triptime'].describe())
print(f"\nTrips longer than 4 hours (240 mins):")
long_trips = trips[trips['triptime'] > 240]
print(f"Count: {len(long_trips)}")
print(long_trips[['tripid','trippurp','linkmode','triptime','travtime',
                   'origlga','destlga']].sort_values('triptime', ascending=False).head(10).to_string())

print(f"\nTrips longer than 8 hours (480 mins):")
very_long = trips[trips['triptime'] > 480]
print(f"Count: {len(very_long)}")
print(very_long[['tripid','trippurp','linkmode','triptime','travtime',
                  'origlga','destlga']].to_string())

print("\n" + "=" * 60)
print("STEP 2 COMPLETE - Share full output for verification")
print("=" * 60)

ISSUE 1: hhinc_group missing values
Rows with missing hhinc_group: 73

Are these missing in the raw file or coded differently?
Unique values in hhinc_group including nulls:
hhinc_group
$2,000-$2,499 ($104,000-$129,999)    129
$500-$649 ($26,000-$33,799)           93
$650-$799 ($33,800-$41,599)           86
$400-$499 ($20,800-$25,999)           77
NaN                                   73
$300-$399 ($15,600-$20,799)           54
$4,500-$4,999 ($234,000-$259,999)     52
$150-$299 ($7,800-$15,599)            23
$1-$149 ($1-$7,799)                   22
$8,000 or more ($416,000 or more)     11
Name: count, dtype: int64

Household sizes of records with missing income:
hhsize
1    43
2    19
3     7
4     4
Name: count, dtype: int64

Home regions of records with missing income:
homeregion_ASGS
Greater Melbourne    66
Geelong SA4           7
Name: count, dtype: int64

ISSUE 2: 60 persons with missing survey weights
Count: 60

Age groups of persons with missing weights:
agegroup
20->24    9
25->

In [6]:
import pandas as pd
import numpy as np
import openpyxl
import re

# ══════════════════════════════════════════════════════════════════
# STEP 3: DATA CLEANING, TRANSFORMATION AND MERGING
# ══════════════════════════════════════════════════════════════════

# ── RELOAD ALL DATASETS ───────────────────────────────────────────

hh    = pd.read_csv("household_vista_2024_2025.csv")
pers  = pd.read_csv("person_vista_2024_2025.csv")
trips = pd.read_csv("trips_vista_2024_2025.csv")
budget = pd.read_csv("state-budget-2023-24-budget-website-map-data.csv")

wb = openpyxl.load_workbook("Local Government Area, Indexes, SEIFA 2021.xlsx", read_only=True)
ws = wb["Table 1"]

rows = []
for i, row in enumerate(ws.iter_rows(values_only=True)):
    if i < 6:
        continue
    if row[0] is None:
        break
    rows.append(row[:11])
seifa = pd.DataFrame(rows, columns=[
    "LGA_Code", "LGA_Name",
    "IRSD_Score", "IRSD_Decile",
    "IRSAD_Score", "IRSAD_Decile",
    "IER_Score",   "IER_Decile",
    "IEO_Score",   "IEO_Decile",
    "Population"
])
seifa["LGA_Code"] = seifa["LGA_Code"].astype(str)
vic_seifa = seifa[seifa["LGA_Code"].str.startswith("2")].copy()

print("All datasets loaded.")
print(f"hh={hh.shape}, pers={pers.shape}, trips={trips.shape}, "
      f"budget={budget.shape}, vic_seifa={vic_seifa.shape}")

# ══════════════════════════════════════════════════════════════════
# HOUSEHOLD CLEANING
# ══════════════════════════════════════════════════════════════════

# C1: hhinc_group — fill NaN with "Not Stated"
# Reason: 73 genuine survey refusals. Retaining rows for travel
# analysis; "Not Stated" keeps them visible rather than silently
# excluded in groupby operations.
hh['hhinc_group'] = hh['hhinc_group'].fillna("Not Stated")
print(f"\nC1: hhinc_group NaN → 'Not Stated': "
      f"{(hh['hhinc_group']=='Not Stated').sum()} rows")

# C2: totalbikes — convert to numeric, recode survey refusals as NaN
# Reason: Column stored as string due to "Missing/Refused" survey code.
hh['totalbikes_clean'] = pd.to_numeric(hh['totalbikes'], errors='coerce')
print(f"C2: totalbikes 'Missing/Refused' → NaN: "
      f"{hh['totalbikes_clean'].isna().sum()} NaNs "
      f"(range {hh['totalbikes_clean'].min():.0f}–"
      f"{hh['totalbikes_clean'].max():.0f})")

# C3: LGA name standardisation for household home LGA
# Reason: VISTA uses "(C)", "(S)", "(B)" suffixes; SEIFA and Budget
# do not. Three additional mismatches require a manual mapping:
# - Merri-bek renamed from Moreland in Nov 2022 (SEIFA predates rename)
# - Bayside and Kingston have "(Vic.)" suffix in SEIFA 2021 to
#   distinguish from same-named LGAs in other states.
def strip_lga_suffix(name):
    """Remove council type suffixes: (C), (S), (B), (RC) etc."""
    return re.sub(r'\s*\([^)]*\)', '', str(name)).strip()

hh['lga_standard'] = hh['homelga'].apply(strip_lga_suffix)

seifa_name_map = {
    'Bayside':   'Bayside (Vic.)',
    'Kingston':  'Kingston (Vic.)',
    'Merri-bek': 'Moreland'
}
hh['lga_for_seifa'] = hh['lga_standard'].replace(seifa_name_map)

seifa_names = set(vic_seifa['LGA_Name'].str.strip())
unmatched = set(hh['lga_for_seifa'].unique()) - seifa_names
print(f"C3: LGA standardised. Unmatched after fix: {unmatched}")

# ══════════════════════════════════════════════════════════════════
# PERSONS CLEANING
# ══════════════════════════════════════════════════════════════════

# C4: numstops — convert to numeric, recode refusals as NaN
# Reason: 60 survey refusals stored as text string.
pers['numstops_clean'] = pd.to_numeric(pers['numstops'], errors='coerce')
print(f"\nC4: numstops refusals → NaN: "
      f"{pers['numstops_clean'].isna().sum()} NaNs")

# Note: 60 persons have missing perspoststratweight.
# These persons recorded zero trips and are children/non-respondents.
# Weights are not used in this EDA so no action required.
# Documented here for transparency.
print(f"NOTE: {pers['perspoststratweight'].isna().sum()} persons with "
      f"missing weights — all have 0 trips recorded. Retained as-is.")

# ══════════════════════════════════════════════════════════════════
# TRIPS CLEANING
# ══════════════════════════════════════════════════════════════════

# C5: starthour normalisation — map hours 24-27 to 0-3
# Reason: VISTA records overnight trips using hours > 23 continuing
# from the previous day (e.g. 25 = 1am next day). This is correct
# VISTA convention but causes axis distortion in time-of-day charts.
# A normalised column maps 24→0, 25→1, 26→2, 27→3.
trips['starthour_norm'] = trips['starthour'].apply(
    lambda h: h - 24 if h > 23 else h)
print(f"\nC5: starthour_norm created. "
      f"Trips with hour > 23: {(trips['starthour'] > 23).sum()} "
      f"(mapped to 0–3 in starthour_norm)")

# C6: cumdist and dist1-dist9 — convert string to float (km)
# Reason: Pure numeric strings. Mean 9.7km, max 381km — valid range
# for Melbourne metropolitan trips including long regional outliers.
trips['cumdist_km'] = pd.to_numeric(trips['cumdist'], errors='coerce')
for col in [f'dist{i}' for i in range(1, 10)]:
    trips[f'{col}_km'] = pd.to_numeric(trips[col], errors='coerce')
print(f"C6: cumdist_km range: "
      f"{trips['cumdist_km'].min():.3f}–{trips['cumdist_km'].max():.3f} km. "
      f"NaNs: {trips['cumdist_km'].isna().sum()}")

# C7: time2-time9 — convert string to numeric (minutes)
# Reason: Trip leg times stored as strings. Pure integers.
for col in [f'time{i}' for i in range(2, 10)]:
    trips[f'{col}_min'] = pd.to_numeric(trips[col], errors='coerce')
print(f"C7: time2-time9 converted to numeric (minutes).")

# C8: LGA name standardisation in trips origlga and destlga
# Reason: 396 trips reference "Merri-bek" in origlga and destlga.
# SEIFA uses the pre-2022 name "Moreland". Applying consistent fix.
# Non-Victorian LGAs (Ballarat, Interstate etc.) are legitimate
# trip origins/destinations — not errors, retained as-is.
trips['origlga_std'] = trips['origlga'].apply(strip_lga_suffix)
trips['destlga_std'] = trips['destlga'].apply(strip_lga_suffix)
trips['origlga_std'] = trips['origlga_std'].replace({'Merri-bek': 'Merri-bek'})
trips['destlga_std'] = trips['destlga_std'].replace({'Merri-bek': 'Merri-bek'})

merri_orig = (trips['origlga_std'] == 'Merri-bek').sum()
merri_dest = (trips['destlga_std'] == 'Merri-bek').sum()
print(f"C8: origlga_std/destlga_std standardised. "
      f"Merri-bek in orig: {merri_orig}, dest: {merri_dest}")

# C9: Trip time outlier flag
# Reason: 5 trips exceed 480 minutes. Four are vehicle driver trips
# within metro Melbourne which is implausible as a single trip.
# One is an interstate flight (plausible). Flagged but retained —
# they represent 0.025% of data and won't affect aggregate analysis.
trips['triptime_flag'] = trips['triptime'] > 480
print(f"C9: {trips['triptime_flag'].sum()} trips flagged as outliers (>480 min)")

# Note on 'duration' column: This records time spent AT the
# destination (activity duration), not travel time. Values of 999
# indicate end of travel diary day. This column is NOT used in
# analysis — triptime is the correct travel time measure.

# ══════════════════════════════════════════════════════════════════
# BUDGET CLEANING
# ══════════════════════════════════════════════════════════════════

# C10: Rename investment column (has trailing space in source data)
budget = budget.rename(columns={'investment ': 'investment'})
print(f"\nC10: Budget 'investment ' column renamed to 'investment'")

# C11: Parse investment to numeric millions
# Reason: Stored as currency string e.g. "$42.2 million*"
# Asterisk (*) indicates bundled funding shared across multiple rows.
# We extract the numeric value but DO NOT sum by LGA for dollar
# totals — bundled rows would cause double-counting. Project COUNT
# per LGA is used instead for Q1 and Q3.
def parse_investment(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().replace('*', '').replace('$', '').replace(',', '')
    val_str = val_str.strip()
    if 'billion' in val_str.lower():
        num = float(re.sub(r'[^\d.]', '',
                           val_str.lower().replace('billion', '')))
        return num * 1000
    elif 'million' in val_str.lower():
        num = float(re.sub(r'[^\d.]', '',
                           val_str.lower().replace('million', '')))
        return num
    else:
        num = float(re.sub(r'[^\d.]', '', val_str))
        return num / 1_000_000

budget['investment_millions'] = budget['investment'].apply(parse_investment)
budget['investment_bundled'] = budget['investment'].str.strip()\
    .str.endswith('*').fillna(False)

print(f"C11: investment parsed. Range: "
      f"${budget['investment_millions'].min():.3f}M–"
      f"${budget['investment_millions'].max():.1f}M. "
      f"Bundled rows: {budget['investment_bundled'].sum()}")

# C12: Spatial flag — rows without coordinates are statewide programs
# Reason: 26 rows have no lat/long (e.g. Free Kinder, Road Maintenance).
# These are policy-level allocations with no specific project location.
# Excluded from spatial Q3 analysis; retained for thematic counts.
budget['has_location'] = budget['lat'].notna()
print(f"C12: Spatial flag. With location: {budget['has_location'].sum()}, "
      f"Without: {(~budget['has_location']).sum()}")

# ══════════════════════════════════════════════════════════════════
# DERIVED DATASET: Investment tier per LGA (for Q1 and Q3)
# ══════════════════════════════════════════════════════════════════

# Tier classification based on education project count per LGA.
# Justification: Using project count (not dollar value) because
# investment_bundled flags mean dollar sums would double-count
# shared funding. Tiers: None=0, Low=1-2, Medium=3-5, High=6+
# based on natural breaks in the distribution (Q1=1, Q3=4).

edu_located = budget[
    (budget['theme'] == 'Education') &
    (budget['has_location'] == True)
].copy()

lga_investment = edu_located.groupby('lga').agg(
    project_count=('unique_id', 'count'),
    project_types=('type', lambda x: list(x.unique()))
).reset_index()

lga_investment.rename(columns={'lga': 'lga_standard'}, inplace=True)

def assign_tier(n):
    if n == 0:   return 'None'
    elif n <= 2: return 'Low (1-2)'
    elif n <= 5: return 'Medium (3-5)'
    else:        return 'High (6+)'

lga_investment['investment_tier'] = \
    lga_investment['project_count'].apply(assign_tier)

# Add "None" rows for VISTA LGAs with zero education projects
all_vista_lgas = hh['lga_standard'].unique()
lgas_with_projects = set(lga_investment['lga_standard'])
lgas_without = [l for l in all_vista_lgas if l not in lgas_with_projects]
none_rows = pd.DataFrame({
    'lga_standard': lgas_without,
    'project_count': 0,
    'project_types': [[] for _ in lgas_without],
    'investment_tier': 'None'
})
lga_investment = pd.concat([lga_investment, none_rows], ignore_index=True)

print(f"\nInvestment tier table created ({len(lga_investment)} LGAs total):")
print(lga_investment['investment_tier'].value_counts())
print("\nHigh-investment LGAs (in VISTA coverage):")
high = lga_investment[
    (lga_investment['investment_tier'] == 'High (6+)') &
    (lga_investment['lga_standard'].isin(all_vista_lgas))
]
print(high[['lga_standard','project_count']].to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# MERGE 1: Full VISTA dataset (trips + persons + households)
# Note: homesubregion_ASGS, homeregion_ASGS, dayType already exist
# in trips — excluded from hh selection to avoid _x/_y suffixes.
# travmonth and travdow come from hh as trips does not have them.
# ══════════════════════════════════════════════════════════════════

vista_merged = trips.merge(
    pers[['persid', 'hhid', 'agegroup', 'sex', 'mainact',
          'anywork', 'anywfh', 'persinc', 'studying',
          'numstops_clean', 'perspoststratweight']],
    on=['persid', 'hhid'], how='left'
).merge(
    hh[['hhid', 'hhsize', 'totalvehs', 'totalbikes_clean',
        'hhinc_group', 'lga_standard', 'lga_for_seifa',
        'travmonth', 'travdow']],
    on='hhid', how='left'
).merge(
    lga_investment[['lga_standard', 'project_count', 'investment_tier']],
    on='lga_standard', how='left'
)

print(f"\nMERGE 1: vista_merged shape: {vista_merged.shape}")
print(f"  Rows preserved: {len(vista_merged) == 19864}")
print(f"  investment_tier nulls: {vista_merged['investment_tier'].isna().sum()}")
print(f"  investment_tier distribution:")
print(vista_merged['investment_tier'].value_counts())

# ══════════════════════════════════════════════════════════════════
# MERGE 2: Households + SEIFA (for Q2)
# ══════════════════════════════════════════════════════════════════

hh_seifa = hh[['hhid', 'lga_standard', 'lga_for_seifa',
               'homesubregion_ASGS', 'homeregion_ASGS',
               'hhinc_group', 'totalvehs', 'totalbikes_clean',
               'hhsize', 'dayType', 'travmonth', 'travdow']].merge(
    vic_seifa[['LGA_Name', 'IRSD_Score', 'IRSD_Decile',
               'IRSAD_Score', 'IRSAD_Decile',
               'IEO_Score',  'IEO_Decile', 'Population']],
    left_on='lga_for_seifa', right_on='LGA_Name', how='left'
)

matched = hh_seifa['IRSD_Score'].notna().sum()
unmatched = hh_seifa['IRSD_Score'].isna().sum()
print(f"\nMERGE 2: hh_seifa shape: {hh_seifa.shape}")
print(f"  Households WITH SEIFA match: {matched} / {len(hh_seifa)}")
print(f"  Households WITHOUT SEIFA match: {unmatched}")
if unmatched > 0:
    print(f"  Unmatched LGAs:")
    print(hh_seifa[hh_seifa['IRSD_Score'].isna()]\
          ['lga_for_seifa'].value_counts())

# ══════════════════════════════════════════════════════════════════
# MERGE 3: Budget spatial education subset (for Q3)
# ══════════════════════════════════════════════════════════════════

budget_spatial = budget[
    (budget['has_location'] == True) &
    (budget['theme'] == 'Education')
].copy()

budget_spatial['lga_standard'] = budget_spatial['lga'].str.strip()

# LGA-level summary for Q3 mapping
lga_q3 = budget_spatial.groupby('lga_standard').agg(
    project_count=('unique_id', 'count'),
    lat_mean=('lat', 'mean'),
    long_mean=('long', 'mean')
).reset_index().merge(
    lga_investment[['lga_standard', 'investment_tier']],
    on='lga_standard', how='left'
).merge(
    hh.groupby('lga_standard').size().reset_index(name='hh_count'),
    on='lga_standard', how='left'
)

print(f"\nMERGE 3: budget_spatial shape: {budget_spatial.shape}")
print(f"  lga_q3 shape: {lga_q3.shape}")
print(f"  lga_q3 sample:")
print(lga_q3.sort_values('project_count', ascending=False)\
      .head(8)[['lga_standard','project_count',
                 'investment_tier','hh_count']].to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# FINAL VALIDATION
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

print(f"\nvista_merged:   {vista_merged.shape}")
print(f"hh_seifa:       {hh_seifa.shape}")
print(f"budget_spatial: {budget_spatial.shape}")
print(f"lga_investment: {lga_investment.shape}")
print(f"lga_q3:         {lga_q3.shape}")

print(f"\nvista_merged — key column null check:")
key_cols = ['trippurp', 'linkmode', 'starthour_norm', 'cumdist_km',
            'lga_standard', 'investment_tier', 'homesubregion_ASGS',
            'homeregion_ASGS', 'agegroup', 'mainact', 'totalvehs',
            'hhinc_group', 'dayType', 'travmonth', 'travdow']
for col in key_cols:
    nulls = vista_merged[col].isna().sum()
    print(f"  {col}: {nulls} nulls")

print(f"\nhh_seifa — SEIFA score null check:")
print(f"  IRSD_Score nulls: {hh_seifa['IRSD_Score'].isna().sum()}")
print(f"  IEO_Score nulls:  {hh_seifa['IEO_Score'].isna().sum()}")

print(f"\nRow counts preserved:")
print(f"  trips original:      19864 → vista_merged: {len(vista_merged)}")
print(f"  households original: 2486  → hh_seifa: {len(hh_seifa)}")

print(f"\nInvestment tier in vista_merged:")
print(vista_merged['investment_tier'].value_counts())

print(f"\nSEIFA decile range in hh_seifa:")
print(f"  IRSD_Decile: "
      f"{hh_seifa['IRSD_Decile'].min()} to "
      f"{hh_seifa['IRSD_Decile'].max()}")

print(f"\nColumns in vista_merged ({len(vista_merged.columns)} total):")
print([c for c in vista_merged.columns if not c.startswith('hhpoststrat')
       and not c.startswith('trippoststrat')
       and not c.startswith('perspoststrat')])

print("\n" + "=" * 60)
print("STEP 3 COMPLETE — Share full output for verification")
print("=" * 60)

All datasets loaded.
hh=(2486, 28), pers=(6404, 53), trips=(19864, 65), budget=(356, 11), vic_seifa=(80, 11)

C1: hhinc_group NaN → 'Not Stated': 73 rows
C2: totalbikes 'Missing/Refused' → NaN: 244 NaNs (range 0–12)
C3: LGA standardised. Unmatched after fix: set()

C4: numstops refusals → NaN: 60 NaNs
NOTE: 60 persons with missing weights — all have 0 trips recorded. Retained as-is.

C5: starthour_norm created. Trips with hour > 23: 80 (mapped to 0–3 in starthour_norm)
C6: cumdist_km range: 0.009–381.529 km. NaNs: 104
C7: time2-time9 converted to numeric (minutes).
C8: origlga_std/destlga_std standardised. Merri-bek in orig: 396, dest: 396
C9: 5 trips flagged as outliers (>480 min)

C10: Budget 'investment ' column renamed to 'investment'
C11: investment parsed. Range: $0.100M–$2800.0M. Bundled rows: 261
C12: Spatial flag. With location: 330, Without: 26

Investment tier table created (67 LGAs total):
investment_tier
Low (1-2)       36
Medium (3-5)    18
High (6+)        8
None        